# Semana 5 · Sesión 1: Derivadas, integrales y aproximaciones

**Módulo 1**

## Objetivos de la sesión

1. Derivar e integrar expresiones simbólicas con `diff` e `integrate`,
   incluyendo derivadas parciales, de orden superior e integrales
   definidas e impropias.
2. Calcular límites con `limit`, incluidos los laterales y los que van al
   infinito, y saber por qué `subs` no los sustituye.
3. Aproximar una expresión con `series` y leer un desarrollo de Taylor
   como lo que es: la física de un régimen límite.

## Retomamos

La semana pasada aprendimos a **construir** expresiones y a reescribirlas:
símbolos con suposiciones, el árbol de `Add`/`Mul`/`Pow`, `expand`,
`factor`, `subs`, `evalf` y `lambdify`.

Todo eso era álgebra. Esta semana empieza el **cálculo**: derivar, integrar,
tomar límites y desarrollar en serie. La diferencia práctica es enorme —
donde antes escribías la fórmula del alcance, ahora puedes *deducirla*.

Este notebook es autocontenido: la celda de abajo declara todo lo que
necesita.

In [ ]:
import sympy as sp

sp.init_printing()

# Símbolos del día, con las suposiciones que el problema físico garantiza.
x, t = sp.symbols("x t", real=True)          # posición y tiempo
r = sp.Symbol("r", positive=True)            # una distancia nunca es negativa
G, M, m = sp.symbols("G M m", positive=True) # constante y masas
k, A, omega = sp.symbols("k A omega", positive=True)

## `diff`: derivar

`sp.diff(expresion, variable)` devuelve la derivada. Como todo en SymPy,
**no modifica** la expresión original: construye una nueva.

Hay tres formas de pedirla, y conviene conocer las tres:

| Quieres | Escribe |
|---|---|
| Derivada de $f$ respecto a $x$ | `sp.diff(f, x)` o `f.diff(x)` |
| Tercera derivada respecto a $x$ | `sp.diff(f, x, 3)` |
| Derivada mixta $\partial^2 f/\partial x\, \partial t$ | `sp.diff(f, x, t)` |

Las derivadas parciales no necesitan sintaxis especial: SymPy trata como
constante todo símbolo que no aparezca en la lista.

In [ ]:
f = sp.sin(x) * sp.exp(-x**2)

print("f      :")
display(f)
print("df/dx  :")
display(sp.diff(f, x))
print("d3f/dx3:")
display(sp.diff(f, x, 3))

# Derivada mixta: k y omega son constantes, x y t no.
onda = sp.sin(k*x) * sp.cos(omega*t)
print("d2/dx dt de una onda:")
display(sp.diff(onda, x, t))

## Derivar sin calcular: `Derivative` y `.doit()`

`sp.diff` calcula de inmediato. A veces conviene lo contrario: escribir la
derivada **sin evaluarla**, para que se vea en la pantalla como se escribe
en el pizarrón, y calcularla después.

Ese objeto es `sp.Derivative`, y `.doit()` es lo que lo ejecuta. El mismo
par existe para integrales (`sp.Integral`), límites (`sp.Limit`) y sumas
(`sp.Sum`): en SymPy, *plantear* y *resolver* son dos pasos separados.

In [ ]:
planteada = sp.Derivative(f, x)

display(planteada)          # se ve como en el pizarrón, sin calcular
display(planteada.doit())   # y aquí sí se calcula

## Física: de un potencial a su fuerza

En una dimensión, la fuerza es menos la derivada del potencial:

$$F(r) = -\frac{dV}{dr}$$

Con el potencial gravitatorio $V(r) = -\dfrac{GMm}{r}$, derivar debe
devolvernos la ley del inverso del cuadrado. Es un buen control: si SymPy
no reprodujera algo que ya sabemos, el error estaría en cómo lo escribimos.

In [ ]:
potencial_gravitatorio = -G*M*m / r

fuerza_gravitatoria = -sp.diff(potencial_gravitatorio, r)

print("V(r):")
display(potencial_gravitatorio)
print("F(r) = -dV/dr:")
display(fuerza_gravitatoria)

## TODO en clase 1

El potencial de un oscilador armónico es $V(x) = \tfrac{1}{2}k x^2$.

1. Escribe `potencial_resorte` con `sp.Rational(1, 2)` para la fracción
   (recuerda la semana 4: un `1/2` de Python es un flotante y contamina).
2. Obtén `fuerza_resorte` como $-dV/dx$ y comprueba que sale la ley de
   Hooke, $F = -kx$.
3. Calcula la **segunda** derivada del potencial. Compárala con la
   constante $k$: ¿qué relación hay?

El punto 3 no es un capricho. $V''$ evaluada en el mínimo es la constante
elástica efectiva de *cualquier* potencial cerca de su equilibrio, y es la
que fija la frecuencia de las oscilaciones pequeñas. Volveremos a ella al
final de la sesión.

In [ ]:
# TODO en clase: el potencial del resorte, su fuerza y su segunda derivada
potencial_resorte = ...

fuerza_resorte = ...

curvatura = ...

## `integrate`: integrar

La misma función hace las dos integrales, y la diferencia está en el
argumento:

| Quieres | Escribe |
|---|---|
| Integral indefinida $\int f\, dx$ | `sp.integrate(f, x)` |
| Integral definida $\int_a^b f\, dx$ | `sp.integrate(f, (x, a, b))` |

Una advertencia que cuesta puntos en los exámenes: **SymPy no escribe la
constante de integración**. La indefinida te devuelve *una* antiderivada,
no la familia completa. El $+C$ corre por tu cuenta.

Los límites pueden ser infinitos: `sp.oo` es el infinito de SymPy (dos
letras o minúsculas, no la letra O).

In [ ]:
print("indefinida (¡sin +C!):")
display(sp.integrate(x**2, x))

print("definida:")
display(sp.integrate(x**2, (x, 0, 1)))

print("impropia, la gaussiana:")
display(sp.integrate(sp.exp(-x**2), (x, -sp.oo, sp.oo)))

## Cuando SymPy no puede

Integrar simbólicamente es mucho más difícil que derivar: hay funciones
elementales cuya antiderivada simplemente **no es** elemental. Cuando
SymPy no encuentra la respuesta no lanza un error — devuelve la integral
**sin evaluar**, un objeto `Integral`.

Eso no es un fracaso: sigue siendo una expresión de SymPy, y siempre puedes
pedirle un número con `evalf`, que la integra numéricamente.

In [ ]:
imposible = sp.integrate(sp.sin(x) / sp.log(x), x)

print("tipo:", type(imposible))
display(imposible)   # se queda planteada: SymPy no la sabe hacer

# Planteada a propósito, y evaluada numéricamente:
numerica = sp.Integral(sp.exp(-x**2), (x, 0, 1))
display(numerica)
print("evalf:", numerica.evalf())

## TODO en clase 2

La **velocidad de escape** sale de igualar la energía cinética inicial al
trabajo necesario para llevar una masa $m$ desde la superficie de un
planeta de radio $R$ hasta el infinito:

$$W = \int_R^{\infty} \frac{GMm}{r^2}\, dr$$

1. Declara `radio` como un símbolo positivo (la $R$ del planeta).
2. Calcula `trabajo_de_escape` con `sp.integrate`, usando `sp.oo` como
   límite superior.
3. Comprueba que coincide con $-V(R)$, es decir con
   `-potencial_gravitatorio.subs(r, radio)`. Úsalo con la receta de
   igualdad matemática de la semana 4, no con `==`.

Que la integral hasta infinito sea **finita** es justo lo que hace posible
escapar de un planeta con una rapidez finita. Un potencial que no decayera
tan rápido no lo permitiría.

In [ ]:
# TODO en clase: el trabajo para escapar de un planeta
radio = ...

trabajo_de_escape = ...

## `limit`: límites de verdad

Para calcular $\lim_{x\to 0}\frac{\sin x}{x}$ no sirve sustituir: `subs`
da $0/0$, que en SymPy es `nan` ("not a number"). El límite es otra
operación, y tiene su propia función: `sp.limit(expresion, variable, punto)`.

Un límite puede además ser **lateral**, y ahí es donde `subs` no tiene nada
que ofrecer. El cuarto argumento lo indica: `"+"` por la derecha (el valor
por defecto), `"-"` por la izquierda.

In [ ]:
cociente = sp.sin(x) / x

print("subs  :", cociente.subs(x, 0))        # nan: indeterminado
print("limit :", sp.limit(cociente, x, 0))   # 1

# Laterales: la misma expresión, dos respuestas distintas.
print("1/x por la derecha  :", sp.limit(1/x, x, 0, "+"))
print("1/x por la izquierda:", sp.limit(1/x, x, 0, "-"))

# Y al infinito, con sp.oo:
print("al infinito         :", sp.limit(sp.exp(-x), x, sp.oo))

## `series`: desarrollo de Taylor

`sp.series(expresion, variable, punto, orden)` desarrolla alrededor de un
punto hasta el orden pedido. Lo que devuelve trae al final un término
`O(x**n)`, que **no es basura**: es la afirmación matemática de que el
error cometido es de ese orden.

Ese término se contagia — si operas con él, todo lo que toque queda
aproximado, que es lo correcto. Cuando quieras quedarte solo con el
polinomio, `.removeO()` lo quita, y ahí sí estás renunciando a la
información sobre el error.

In [ ]:
desarrollo = sp.series(sp.sin(x), x, 0, 8)

display(desarrollo)                # con el término O(x**8)
display(desarrollo.removeO())      # solo el polinomio

## Una serie es un régimen físico

Aquí está la razón por la que un físico usa `series`: cada truncamiento es
una **aproximación con nombre propio**.

La energía relativista de una partícula es
$E = \dfrac{mc^2}{\sqrt{1 - v^2/c^2}}$. Desarrollarla en potencias de la
rapidez, alrededor de $v = 0$, tiene que devolver la energía en reposo más
la energía cinética de siempre — más correcciones cada vez más pequeñas.

Si el desarrollo no reprodujera $\tfrac{1}{2}mv^2$, alguna de las dos
teorías estaría mal.

In [ ]:
c, v = sp.symbols("c v", positive=True)

energia_relativista = m*c**2 / sp.sqrt(1 - v**2/c**2)

# Desarrollo en la rapidez, alrededor del reposo.
display(sp.series(energia_relativista, v, 0, 6))

El primer término es $mc^2$, la energía en reposo. El segundo es
$\tfrac{1}{2}mv^2$: la energía cinética newtoniana **aparece sola**, como
la primera corrección al reposo. El tercero, $\tfrac{3}{8}mv^4/c^2$, es la
primera corrección relativista — la que empieza a importar cuando $v$ deja
de ser pequeña frente a $c$.

Esto es lo que quiere decir que la mecánica newtoniana sea el límite de
baja velocidad de la relativista, escrito como una cuenta que puedes hacer
en un renglón.

## TODO en clase 3

La energía potencial de un péndulo simple, medida desde el punto más bajo,
es

$$V(\theta) = m g L\,(1 - \cos\theta)$$

1. Declara `theta` como símbolo real, y `longitud` y `gravedad` como
   positivos.
2. Escribe `potencial_pendulo` y desarróllalo en serie alrededor de
   $\theta = 0$ hasta orden 5.
3. Quédate con el primer término no nulo (`removeO`, y luego mira el
   resultado): es $\tfrac{1}{2} m g L\, \theta^2$.
4. Compáralo con el potencial del resorte del TODO 1. Para que la
   comparación sea directa hay que usar la misma coordenada que él: una
   **longitud**, no un ángulo. La longitud de arco es $s = L\theta$, así
   que reescribe el término como $\tfrac{1}{2}\,k_{\text{ef}}\, s^2$ y lee
   la constante elástica efectiva $k_{\text{ef}}$.
5. Con esa $k_{\text{ef}}$, la masa $m$ y la relación
   $\omega = \sqrt{k/m}$ del oscilador armónico, escribe la frecuencia del
   péndulo. Debe salirte $\omega = \sqrt{g/L}$ — la fórmula de la
   aproximación de ángulos pequeños, deducida en lugar de recordada.

Fíjate en el orden: pedimos hasta 5 pero el desarrollo salta de $\theta^2$
a $\theta^4$. Los términos impares son cero porque el coseno es par, y eso
también es física: el potencial es simétrico ante $\theta \to -\theta$.

In [ ]:
# TODO en clase: el péndulo en la aproximación de ángulos pequeños
theta = ...
longitud = ...
gravedad = ...

potencial_pendulo = ...

desarrollo_pendulo = ...

## Resumen

Hoy pasamos del álgebra al cálculo. `diff` deriva —parciales y de orden
superior sin sintaxis extra— y `integrate` integra, con la advertencia de
que la indefinida **no trae** el $+C$ y de que a veces devuelve la integral
sin evaluar, porque el problema es genuinamente difícil. Vimos también el
par plantear/resolver: `Derivative`, `Integral` y su `.doit()`.

`limit` hace lo que `subs` no puede, incluidos los límites laterales y al
infinito. Y `series` convierte una expresión en la física de un régimen:
el péndulo se vuelve armónico y la energía relativista se vuelve
newtoniana, cada una en su límite.

**Próxima sesión — Semana 5, sesión 2:** el otro lado del cálculo
simbólico. Resolver ecuaciones (`solve`, `solveset`, `linsolve`,
`nonlinsolve`) y, con `dsolve`, ecuaciones diferenciales — para pasar de
las fuerzas de hoy a las trayectorias que producen.